## Load the download function

In [9]:
import pandas as pd
import requests
import json
import re


def gn_antcat_download(search_term, filter_terms):
    """
    Query AntCat GeoNetwork for records matching a controlled vocabulary
    search term and optional filter terms, then download and combine
    the associated datasets.

    Parameters:
        search_term  : str        - Primary keyword (controlled vocabulary)
        filter_terms : list[str]  - Additional keyword filters

    Returns:
        records      : pd.DataFrame - Metadata records (uuid, title, data_link)
        data         : pd.DataFrame - Combined downloaded datasets
    """

    BASE    = "https://antcat.antarcticanz.govt.nz/geonetwork"
    URL     = f"{BASE}/srv/api/search/records/_search?bucket=metadata&relatedType=datasets"
    HEADERS = {"accept": "application/json", "Content-Type": "application/json"}

    # ------------------------------------------------------------------
    # 1. Build request payload
    # ------------------------------------------------------------------
    def build_payload(search_term, filter_terms):
        filter_dict = {}

        payload = {
            "query": {
                "bool": {
                    "must": [
                        {
                            "term": {
                                "tag.default": {"value": search_term}
                            }
                        }
                    ],
                    "filter": {}
                }
            },
            "_source": {
                "includes": ["uuid", "linkUrl", "resourceTitleObject"]
            },
            "size": 100
        }

        for i, filter_term in enumerate(filter_terms):
            if filter_term != "":
                key = "term" if i == 0 else f"term.{i}"
                filter_dict[key] = {"tag.default": {"value": filter_term}}

        if filter_dict:
            payload["query"]["bool"]["filter"] = filter_dict

        return json.dumps(payload, indent=2)

    # ------------------------------------------------------------------
    # 2. Download and parse a single dataset (in-memory)
    # ------------------------------------------------------------------
    def fetch_dataset(metadata_url, data_title, data_link):
        if not data_link:
            print(f"  [SKIP] No data link for: {data_title}")
            return {"data": None, "problems": "No data link provided", "id": metadata_url}

        try:
            print(f"  [DOWNLOAD] {data_title}")
            response = requests.get(data_link)
            response.raise_for_status()

            # Decode content and find where data starts (after */ header)
            lines = response.content.decode("utf-8").splitlines()
            pattern = r"\*/"
            position = next((i for i, line in enumerate(lines) if re.search(pattern, line)), None)
            position = (position + 1) if position is not None else 0

            from io import StringIO
            content = "\n".join(lines[position:])
            df = pd.read_csv(StringIO(content), sep="\t")
            df["metadata_url"] = metadata_url

            print(f"  [OK] {len(df)} rows")
            return {"data": df, "problems": None, "id": metadata_url}

        except Exception as e:
            print(f"  [ERROR] {data_title}: {e}")
            return {"data": None, "problems": e, "id": metadata_url}

    # ------------------------------------------------------------------
    # 3. Query the API
    # ------------------------------------------------------------------
    print(f"Querying AntCat for: '{search_term}' with filters: {filter_terms}")
    payload = build_payload(search_term, filter_terms)

    resp = requests.post(URL, headers=HEADERS, data=payload)
    resp.raise_for_status()
    print(f"Response status: {resp.status_code}")

    parsed   = json.loads(resp.text)
    hits     = parsed["hits"]["hits"]
    print(f"Total records found: {len(hits)}")

    # ------------------------------------------------------------------
    # 4. Build records DataFrame
    # ------------------------------------------------------------------
    extracted = []
    for item in hits:
        title = item["_source"]["resourceTitleObject"]["default"]
        extracted.append({
            "metadata_url": f"{BASE}/srv/eng/catalog.search#/metadata/{item['_id']}",
            "data_title":   re.sub(r"[^a-zA-Z0-9-]", " ", title),
            "data_link":    item["_source"].get("linkUrl")
        })

    records = pd.DataFrame(extracted)
    print(f"\nRecords DataFrame ({len(records)} rows):")
    print(records)

    # ------------------------------------------------------------------
    # 5. Download all datasets
    # ------------------------------------------------------------------
    print(f"\nDownloading {len(records)} datasets...")
    results = records.apply(
        lambda row: fetch_dataset(
            metadata_url=row["metadata_url"],
            data_title=row["data_title"],
            data_link=row["data_link"]
        ),
        axis=1
    )

    # ------------------------------------------------------------------
    # 6. Combine into single DataFrame
    # ------------------------------------------------------------------
    frames = [r["data"] for r in results if r["data"] is not None]

    if frames:
        data = pd.concat(frames, ignore_index=True)
        print(f"\nCombined data DataFrame ({len(data)} rows, {len(data.columns)} columns):")
        print(data.head())
    else:
        data = pd.DataFrame()
        print("\nNo data successfully downloaded.")

    return records, data

## Run the query

In [10]:
#from gn_antcat_download import gn_antcat_download

records, drill_hole_data = gn_antcat_download(
    search_term="DRILL HOLE",
    filter_terms=["ICE DEPTH/THICKNESS", "SUB-ICE PLATELET LAYER"]
)

Querying AntCat for: 'DRILL HOLE' with filters: ['ICE DEPTH/THICKNESS', 'SUB-ICE PLATELET LAYER']
Response status: 200
Total records found: 6

Records DataFrame (6 rows):
                                                                                                           metadata_url  \
0  https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/7c510bd1-9afa-40be-b64c-480da0e96fab   
1  https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/21ad7d35-4702-43eb-9efb-f813aa7eef82   
2  https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/77f281f9-1bad-45af-aa3a-9abaa50fc5bc   
3  https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/663a5a84-dad3-4cfd-a53d-dcbcae33934a   
4  https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/d3479d00-5f8a-43e1-bacf-be3e40a82f85   
5  https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/3b4cbdaf

## Inspect the data

In [11]:
import pandas as pd
pd.set_option("display.max_colwidth", None)  # show full column contents

drill_hole_data

,Event,Date/Time,Longitude,Latitude,ID,EsEs [m],SIPL [m],Freeboard [m],Snow thick [m],metadata_url
0,McMurdo-2011_1,2011-11-23,166.400,-77.7667,1,1.96,0.22,0.06,0.34,https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/7c510bd1-9afa-40be-b64c-480da0e96fab
1,McMurdo-2011_2,2011-11-23,166.200,-77.7667,2,1.98,0.46,0.06,0.34,https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/7c510bd1-9afa-40be-b64c-480da0e96fab
2,McMurdo-2011_3,2011-11-23,166.000,-77.7667,3,2.12,0.88,0.11,0.24,https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/7c510bd1-9afa-40be-b64c-480da0e96fab
3,McMurdo-2011_5,2011-11-24,166.200,-77.6667,4,1.51,0.00,0.01,0.36,https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/7c510bd1-9afa-40be-b64c-480da0e96fab
4,McMurdo-2011_6,2011-11-24,166.000,-77.6667,5,1.57,0.00,0.05,0.29,https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/7c510bd1-9afa-40be-b64c-480da0e96fab
...,...,...,...,...,...,...,...,...,...,...
148,McMurdo-2018_23,2018-11-12,166.000,-77.8333,10,3.37,1.50,0.29,0.23,https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/3b4cbdaf-3912-49ea-9601-107d7436dedc
149,McMurdo-2018_3,2018-11-13,166.029,-77.7745,11,2.05,0.97,0.15,0.19,https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/3b4cbdaf-3912-49ea-9601-107d7436dedc
150,McMurdo-2018_4,2018-11-13,165.600,-77.7667,12,2.22,4.71,0.27,0.11,https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/3b4cbdaf-3912-49ea-9601-107d7436dedc
151,McMurdo-2018_100,2018-11-19,166.655,-77.8673,13,2.49,1.46,0.16,0.38,https://antcat.antarcticanz.govt.nz/geonetwork/srv/eng/catalog.search#/metadata/3b4cbdaf-3912-49ea-9601-107d7436dedc


## Plot/Map the data

In [5]:
from folium import Map,CircleMarker
from IPython.display import IFrame,display
import base64


def show_folium_safe(m : Map, height=500):
    html_content = m.get_root().render()
    encoded = base64.b64encode(html_content.encode('utf-8')).decode('utf-8')
    src = f"data:text/html;base64,{encoded}"
    display(IFrame(src, width='100%', height=height))

m = Map(location=[-77.55, 166.4], zoom_start=7, tiles='OpenStreetMap')

for _, row in drill_hole_data.iterrows():
    CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5,
        color='red',
        fill=True,
        fill_color='blue',
        fill_opacity=0.7
    ).add_to(m)

show_folium_safe(m)